# 01 · 什么是"学习"？从线性回归说起

> **本节属于 Part 1 · 数学与梯度直觉。** 这是整个项目的第一块基石。

深度学习里所有的"训练"，本质上都在重复同一个循环：

$$\textbf{模型预测} \rightarrow \textbf{算损失} \rightarrow \textbf{求梯度} \rightarrow \textbf{更新参数} \rightarrow \cdots$$

我们用最简单的**线性回归**（拟合一条直线）把这个循环彻底讲透。一旦你理解了它，后面的 MLP、CNN、Transformer 都只是"同一个循环、更复杂的模型"而已。

## 学习目标

- 理解机器"学习"的本质 = **用梯度下降最小化损失**
- 亲手实现线性回归的**前向、损失(MSE)、梯度、参数更新**四件套
- 画出**拟合过程**与**损失下降曲线**
- 用**解析解（正规方程）**、**数值梯度检查**、**PyTorch** 三种方式验证我们的结果

## 直觉与数学原理

**问题**：给定一堆数据点 $(x_i, y_i)$，我们想找一条直线 $\hat{y} = wx + b$ 来拟合它们。

**模型**：$\hat{y} = wx + b$，其中 $w$（斜率）和 $b$（截距）是待学习的**参数**。

**损失函数**：用**均方误差 (MSE)** 衡量"拟合得有多差"：

$$L(w,b) = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2 = \frac{1}{n}\sum_{i=1}^{n}(wx_i + b - y_i)^2$$

损失越小，拟合越好。所以**"学习" = 寻找让 $L$ 最小的 $w, b$**。

**怎么找？梯度下降。** 梯度 $\nabla L$ 指向损失"上升最快"的方向，那么沿它的**反方向**走一小步，损失就会下降：

$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w}, \qquad b \leftarrow b - \eta\,\frac{\partial L}{\partial b}$$

其中 $\eta$ 是**学习率**（步长）。对 MSE 求偏导（用链式法则，下一节细讲），得到：

$$\frac{\partial L}{\partial w} = \frac{1}{n}\sum_i 2(\hat{y}_i - y_i)\,x_i, \qquad \frac{\partial L}{\partial b} = \frac{1}{n}\sum_i 2(\hat{y}_i - y_i)$$

接下来我们把这几个公式**一行行用 NumPy 写出来**。

## 从零手写实现

### 第一步：造一批数据

我们人为设定"真实规律" $y = 2x - 1$，再加一点噪声，模拟现实中带噪的观测。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from minitorch import set_seed

set_seed(42)
true_w, true_b = 2.0, -1.0                       # 真实规律（学习的"答案"）
x = np.random.uniform(-3, 3, size=100)
y = true_w * x + true_b + np.random.randn(100) * 0.5   # 加噪声

plt.figure(figsize=(5, 3.5))
plt.scatter(x, y, s=14, alpha=0.7)
plt.xlabel("x"); plt.ylabel("y")
plt.title("Data to fit (true rule: y = 2x - 1)")   # 图内文字用英文，避免缺字体
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### 第二步：实现"四件套"并训练

注意下面循环里的四行核心代码，它就是**一切深度学习训练的雏形**。

In [ ]:
def forward(w, b, x):
    """模型：给定参数和输入，算出预测值。"""
    return w * x + b

def mse_loss(yhat, y):
    """损失：均方误差。"""
    return np.mean((yhat - y) ** 2)

# 参数从 0 开始（故意从"错"的地方出发，看它怎么学到正确答案）
w, b = 0.0, 0.0
lr, epochs = 0.05, 300
history = []

for epoch in range(epochs):
    yhat = forward(w, b, x)                  # ① 前向：算预测
    loss = mse_loss(yhat, y)                 # ② 算损失
    grad_w = np.mean(2 * (yhat - y) * x)     # ③ 求梯度（手推公式）
    grad_b = np.mean(2 * (yhat - y))
    w -= lr * grad_w                         # ④ 更新参数（沿负梯度走一步）
    b -= lr * grad_b
    history.append(loss)

print(f"学到的参数 : w = {w:.3f},  b = {b:.3f}")
print(f"真实参数   : w = {true_w},  b = {true_b}")
print(f"最终损失   : {history[-1]:.4f}")

### 第三步：看看学得怎么样

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

axes[0].scatter(x, y, s=14, alpha=0.6, label="data")
xs = np.array([x.min(), x.max()])
axes[0].plot(xs, forward(w, b, xs), "r-", lw=2, label=f"fit: y={w:.2f}x+{b:.2f}")
axes[0].legend(); axes[0].set_title("Fit result"); axes[0].grid(True, alpha=0.3)

axes[1].plot(history)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("MSE"); axes[1].set_yscale("log")
axes[1].set_title("Loss curve (log scale)"); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 验证

我们用两种独立的方式确认结果正确。

### 验证一：与解析解（正规方程）对比

线性回归其实有**闭式解**——直接用最小二乘可以一步算出最优 $w, b$，无需迭代。如果我们的梯度下降是对的，结果应当与它非常接近。

In [ ]:
# 把截距 b 看成"常数特征 1"的权重： X = [x, 1]，解 min ||Xβ - y||^2
X = np.stack([x, np.ones_like(x)], axis=1)        # shape (n, 2)
(w_closed, b_closed) = np.linalg.lstsq(X, y, rcond=None)[0]

print(f"解析解(正规方程) : w = {w_closed:.3f}, b = {b_closed:.3f}")
print(f"梯度下降结果     : w = {w:.3f}, b = {b:.3f}")
print("两者接近 ->", np.allclose([w, b], [w_closed, b_closed], atol=1e-2))

### 验证二：数值梯度检查

我们"手推"的梯度公式对不对？用 `minitorch.gradcheck`（上一节介绍过的中心差分工具）来验收：把损失看成参数向量 $p=[w,b]$ 的函数，比较**手推梯度**与**数值梯度**。

In [ ]:
from minitorch import gradcheck

def loss_of_params(p):
    return mse_loss(forward(p[0], p[1], x), y)

p0 = np.array([0.5, -0.3])                        # 任取一个检查点
yhat0 = forward(p0[0], p0[1], x)
analytic = np.array([np.mean(2 * (yhat0 - y) * x), np.mean(2 * (yhat0 - y))])

gradcheck(loss_of_params, p0, analytic, name="线性回归手推梯度")

## PyTorch 对照

同样的任务，用 PyTorch 的 `nn.Linear` + `SGD` 怎么写？注意：**循环结构和我们手写的一模一样**（前向 → 损失 → backward → step），只是梯度由 PyTorch 自动求出。这正是后面 Part 2/3 我们要亲手造的"自动求导"。

In [ ]:
import torch
import torch.nn as nn

set_seed(42)
xt = torch.tensor(x, dtype=torch.float32).reshape(-1, 1)
yt = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

model = nn.Linear(1, 1)                            # 等价于 y = wx + b
opt = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

for epoch in range(300):
    opt.zero_grad()
    loss = loss_fn(model(xt), yt)
    loss.backward()                                # PyTorch 自动求梯度
    opt.step()

print(f"PyTorch 学到 : w = {model.weight.item():.3f}, b = {model.bias.item():.3f}")
print(f"我们手写的   : w = {w:.3f}, b = {b:.3f}")

## 📦 沉淀进 minitorch

本节是**原理铺垫**，目的是建立"模型 / 损失 / 梯度 / 更新"这个核心直觉，暂时没有可复用的代码需要沉淀进包。

真正的"造轮子"从 **Part 2（`03_value_autograd_engine`）** 开始——我们将造一个能**自动**求梯度的引擎，从此告别"手推梯度"。

## 小练习

1. **学习率的威力**：把 `lr` 改成 `1.0` 再跑一次，观察损失曲线——它会震荡甚至发散吗？再试 `lr=0.001`，又会怎样（提示：收敛很慢）？体会学习率为什么是最重要的超参数之一。
2. **加大噪声**：把噪声标准差从 `0.5` 改成 `2.0`，重新生成数据训练。学到的 `w, b` 还接近真值吗？拟合直线变了吗？
3. **多元线性回归（进阶）**：把 `x` 换成两维特征（`shape=(100, 2)`），真实规律设为 $y = 3x_1 - 2x_2 + 1$，把 `w` 改成长度 2 的向量，用 `yhat = x @ w + b` 重写前向，并相应修改梯度公式。用正规方程验证你的结果。

## 小结 & 下一站

✅ 我们亲手实现了贯穿所有深度学习的核心循环：**前向 → 损失 → 梯度 → 更新**，并用三种方式验证了正确性。

但有个问题：这里的梯度是我们**手推**出来的。模型一旦变复杂（多层、非线性），手推梯度会变得极其繁琐且容易出错。

**下一站 → `02_chain_rule_and_manual_backprop`**：我们先把"链式法则"和"反向传播"彻底搞懂，手推一个两层网络的梯度——这会让你深刻体会到：为什么我们迫切需要一个**自动**求导的引擎。